- 배포 모델 확정 실험 — 정리 데이터(201,616) · 레시피(eff128 · lr 4.8e-4) · **max_len 4096**
- 비교선: 앵커 `11_01` 0.8588(정리 test) ~ exp1 0.8683(정리 test 재계산, 구 레시피 8192) 사이
- 4096 선택 근거: 절단 문서 0.8% · 유실 토큰 1.60%(8192는 0.02%)이고 최악 시퀀스가 1/2이라 micro_batch를 2배로 써 훈련 시간을 단축할 수 있음.
- lr 안전: LR range test 전환점 512 8.84e-3 · 8192 7.13e-3 → 4096은 그 사이, 운영 lr이 15배 아래

In [1]:
import sys
sys.path.insert(0, "src")  # /workspace/src

from patent_train import TrainingRunner, TrainConfig, probe_batches

In [9]:
SEARCH = False   # True=fast-fail 짧은 런(2 epoch로 유도) / False=풀런(아래 epochs 사용)

cfg = TrainConfig(
    backbone="axenc",       # backbones.BACKBONES 키
    loss="focal",
    loss_params={"alpha": 0.25, "gamma": 2},
    max_len=4096,
    eff_batch=128,          # 배치 재현
    micro_batch=16,        
    eval_micro_batch=128,
    learning_rate=4.8e-4,   
    weight_decay=0.01,
    warmup_ratio=0.1,
    epochs=12,
    early_stop_epochs=2,    
    notebook_name="16_01_Model_4096.ipynb",   # wandb code saving
    tag="modernbert-patent-len4096-op",
    run_name="axenc_len4096_focal_op",
    repo_final="ingyoun/A.X-patent-len4096-op",
    out_path="/workspace/output/modernbert-len4096-op",
    search=SEARCH,
)

print("run_name:", cfg.run_name, "| epochs:", cfg.epochs, "| grad_accum:", cfg.grad_accum)

run_name: axenc_len4096_focal_op | epochs: 12 | grad_accum: 8


## 구성 — 데이터·모델

토크나이저+원본 로드 → `max_len` 절단(캐시) → 분류기 구성. 단계를 나눠 중간 점검·부분 재실행이 가능하다.

In [10]:
runner = TrainingRunner(cfg)

In [11]:
runner.load_data()        # 토크나이저 + 원본 데이터셋(prep 캐시 있으면 원본 생략)
runner.data.raw

[skip] prep 캐시 존재 — 원본 로드 생략: /workspace/prep_cache/axenc_len4096


In [12]:
runner.prepare_data()     # max_len 절단 → prep 캐시
runner.data.dataset

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 201616
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11244
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11132
    })
})

In [13]:
runner.load_model()

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: skt/A.X-Encoder-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[model] skt/A.X-Encoder-base@9708f9c4(신규 헤드)


## OOM 확인

GPU/배치에서 안전한 `micro_batch` 상한을 실측

In [8]:
probe_batches(
    runner.model.to("cuda"), runner.data.tokenizer.vocab_size, cfg.max_len,
    train_mb=(4, 8, 16, 32, 64, 128),
    eval_mb=(8, 16, 32, 64, 96, 128, 160, 192, 224),
)

train micro=  4: peak  12.2 GB  OK
train micro=  8: peak  22.4 GB  OK
train micro= 16: peak  42.7 GB  OK
train micro= 32: OOM
eval  micro=  8: peak   2.9 GB  OK
eval  micro= 16: peak   3.8 GB  OK
eval  micro= 32: peak   5.7 GB  OK
eval  micro= 64: peak   9.3 GB  OK
eval  micro= 96: peak  12.9 GB  OK
eval  micro=128: peak  16.5 GB  OK
eval  micro=160: peak  20.2 GB  OK
eval  micro=192: peak  23.8 GB  OK
eval  micro=224: peak  27.4 GB  OK


## 훈련

In [14]:
runner.build_trainer()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


[schedule] 1576 step/epoch | eval·save 788 step마다(2회/epoch) | early stop 2 epoch(patience=4 eval)


In [15]:
runner.train()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
788,0.001417,0.000916,0.623720,0.571428,0.540405,0.338304,0.661509
1576,0.000889,0.000642,0.735452,0.716211,0.712020,0.119745,0.717555
2364,0.000626,0.000529,0.770757,0.749263,0.751043,0.109324,0.760677
3152,0.000544,0.000498,0.789616,0.775638,0.779093,0.086777,0.772712
3940,0.000479,0.000475,0.801127,0.792732,0.793862,0.071506,0.776030
4728,0.000494,0.000462,0.801592,0.791136,0.797781,0.067643,0.778991
5516,0.000447,0.000448,0.811911,0.803148,0.811190,0.053989,0.783798
6304,0.000437,0.000442,0.808934,0.800400,0.797749,0.077524,0.791602
7092,0.000378,0.000440,0.822313,0.815719,0.827408,0.042490,0.794019
7880,0.000384,0.000421,0.825827,0.819960,0.830216,0.042580,0.797376


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 평가 · 메트릭 저장 · push

모델 가중치는 로컬에 두지 않고 Hub로만 올린다(팟을 지우면 로컬 사본은 사라진다). 로컬 사본이 필요하면 `runner.save_model()`.

In [16]:
test_metrics = runner.evaluate("test")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

[transformers] early stopping required metric_for_best_model, but did not find eval_micro_f1 so early stopping is disabled


Training Loss,Validation Loss,Step,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
0.000020,0.000795,18912,0.866034,0.863779,0.883506,0.008271,0.825135


test_loss: 0.0007953844033181667
test_micro_f1: 0.8660344890711383
test_macro_f1: 0.8637787673728732
test_sample_f1: 0.8835061135354624
test_empty_rate: 0.008271077908217716
test_anchor_weighted_f1: 0.825135139892277


In [17]:
runner.save_metrics()     # runner.metrics(split 전체) → {tag}_metrics.json

[save] /workspace/output/modernbert-len4096-op/modernbert-patent-len4096-op_metrics.json  splits=['test']


In [18]:
runner.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

[push] ingyoun/A.X-patent-len4096-op


## val·test 로짓 덤프

`logits_{tag}_{split}.npy`를 `out_path` 상위(`/workspace/output/`)에 저장

In [19]:
runner.predict_logits("val")
runner.predict_logits("test")

[dump] /workspace/output/logits_modernbert-patent-len4096-op_val.npy  shape=(11132, 188)


[dump] /workspace/output/logits_modernbert-patent-len4096-op_test.npy  shape=(11244, 188)


array([[ 3.015625 , -6.03125  , -7.03125  , ..., -6.90625  , -6.84375  ,
        -7.53125  ],
       [ 0.8046875, -5.28125  , -7.875    , ..., -6.5625   , -5.5625   ,
        -6.53125  ],
       [ 1.671875 , -5.6875   , -4.53125  , ..., -5.15625  , -3.84375  ,
        -6.03125  ],
       ...,
       [-7.75     , -8.25     , -8.0625   , ..., -6.59375  , -7.75     ,
         6.65625  ],
       [-7.625    , -8.5      , -7.8125   , ..., -7.21875  , -8.       ,
         6.4375   ],
       [-7.75     , -8.5625   , -8.0625   , ..., -7.1875   , -8.375    ,
         5.40625  ]], shape=(11244, 188), dtype=float32)